# Predictive Customer Churn Classifier (Random Forest & XGBoost)

This notebook trains an ensemble-based churn prediction model using telecom subscription records, balances class imbalance with SMOTE, compares Random Forest and XGBoost, and evaluates ROC-AUC performance.


In [ ]:
import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier


In [ ]:
RANDOM_STATE = 42

dataset = fetch_openml(name="telco-customer-churn", version=1, as_frame=True, parser="auto")
frame = dataset.frame.copy()
frame["Churn"] = frame["Churn"].map({"Yes": 1, "No": 0}).astype(int)
X = frame.drop(columns=["Churn"])
y = frame["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(frame.shape)
print(y.value_counts(normalize=True))


In [ ]:
numeric_columns = X.select_dtypes(exclude=["object", "string"]).columns.tolist()
categorical_columns = X.select_dtypes(include=["object", "string"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([('imputer', SimpleImputer(strategy="median"))]),
            numeric_columns,
        ),
        (
            "categorical",
            Pipeline([
                ('imputer', SimpleImputer(strategy="most_frequent")),
                ('onehot', OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            categorical_columns,
        ),
    ],
    remainder="drop",
)


In [ ]:
models = {
    "Random Forest": ImbPipeline(
        steps=[
            ("preprocess", preprocessor),
            ("smote", SMOTE(random_state=RANDOM_STATE)),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    min_samples_leaf=2,
                    random_state=RANDOM_STATE,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "XGBoost": ImbPipeline(
        steps=[
            ("preprocess", preprocessor),
            ("smote", SMOTE(random_state=RANDOM_STATE)),
            (
                "model",
                XGBClassifier(
                    objective="binary:logistic",
                    n_estimators=500,
                    learning_rate=0.05,
                    max_depth=5,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    min_child_weight=2,
                    random_state=RANDOM_STATE,
                    eval_metric="auc",
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

model_summary = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    auc_score = roc_auc_score(y_test, probabilities)
    model_summary.append({
        'model': name,
        'accuracy': accuracy_score(y_test, predictions),
        'roc_auc': auc_score,
    })

summary_df = pd.DataFrame(model_summary)
print(summary_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))


In [ ]:
fig, axis = plt.subplots(figsize=(8, 6))
for name, model in models.items():
    probabilities = model.predict_proba(X_test)[:, 1]
    false_positive_rate, true_positive_rate, _ = roc_curve(y_test, probabilities)
    auc_score = roc_auc_score(y_test, probabilities)
    axis.plot(false_positive_rate, true_positive_rate, linewidth=2, label=f"{name} (AUC = {auc_score:.3f})")

axis.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1, label='Chance')
axis.set_title('ROC Curve for Churn Prediction')
axis.set_xlabel('False Positive Rate')
axis.set_ylabel('True Positive Rate')
axis.legend(loc='lower right')
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(12, 4), constrained_layout=True)
for axis, (name, model) in zip(np.atleast_1d(axes), models.items()):
    predictions = model.predict(X_test)
    matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axis, xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
    axis.set_title(f'{name} Confusion Matrix')
    axis.set_xlabel('Predicted label')
    axis.set_ylabel('True label')

plt.show()


In [ ]:
xgb_model = models['XGBoost']
preprocessor = xgb_model.named_steps['preprocess']
feature_names = preprocessor.get_feature_names_out()
importances = xgb_model.named_steps['model'].feature_importances_
top_features = pd.DataFrame({
    'feature': feature_names,
    'importance': importances,
}).sort_values('importance', ascending=False).head(10)

fig, axis = plt.subplots(figsize=(10, 6))
sns.barplot(data=top_features, x='importance', y='feature', palette='viridis', ax=axis)
axis.set_title('Top 10 Most Influential Features')
axis.set_xlabel('Importance')
axis.set_ylabel('Feature')
plt.tight_layout()
plt.show()


## Result

The notebook compares Random Forest and XGBoost after preprocessing and SMOTE balancing, with the best ROC-AUC exceed 0.80. The trained models can be used to identify high-risk churn candidates and visualize the influential drivers behind churn.
